In [1]:
# MÓDULO 1 — PREPARAÇÃO DO AMBIENTE E DEFINIÇÃO DO RECORTE
# Define as temporadas, as pastas e as bibliotecas utilizadas.

from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    leaguedashplayerbiostats,
    shotchartdetail
)
from requests.exceptions import RequestException
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import time
import sys

temporadas = [
    f"{ano}-{str(ano + 1)[-2:]}"
    for ano in range(2015, 2025)
]

pasta_jogadores = Path("dados/jogadores_por_temporada")
pasta_arremessos = Path("dados/arremessos")
pasta_perfis = Path("dados/perfis_por_temporada")
pasta_auditorias = Path("dados/auditorias")

for pasta in [pasta_jogadores, pasta_arremessos, pasta_perfis]:
    pasta.mkdir(parents=True, exist_ok=True)

print("Python utilizado:", sys.executable)
print("Temporadas:", temporadas)

Python utilizado: c:\Users\felip\Desktop\teste-nba\.venv\Scripts\python.exe
Temporadas: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


In [2]:
# MÓDULO 2 — COLETA DAS ESTATÍSTICAS DOS JOGADORES POR TEMPORADA
# Reutiliza arquivos existentes ou consulta a API com até três tentativas.
# Valida a estrutura e as chaves da base que servirá de referência.

tabelas_jogadores = []

colunas_estatisticas = {
    "PLAYER_ID", "PLAYER_NAME", "SEASON",
    "FGA", "FGM", "FG3A", "FG3M"
}

for temporada in temporadas:
    arquivo = pasta_jogadores / f"{temporada}.csv"
    arquivo_novo = not arquivo.exists()

    if arquivo_novo:
        for tentativa in range(1, 4):
            time.sleep(5)

            try:
                resposta = leaguedashplayerstats.LeagueDashPlayerStats(
                    season=temporada,
                    season_type_all_star="Regular Season",
                    per_mode_detailed="Totals",
                    timeout=60
                )

                tabela = resposta.league_dash_player_stats.get_data_frame()
                tabela["SEASON"] = temporada
                break

            except (RequestException, ValueError) as erro:
                print(f"{temporada} | tentativa {tentativa}/3: {erro}")

                if tentativa == 3:
                    raise

                time.sleep(15 * tentativa)
    else:
        tabela = pd.read_csv(arquivo)

    faltantes = colunas_estatisticas - set(tabela.columns)

    if faltantes or tabela.empty:
        raise ValueError(
            f"{temporada}: tabela vazia ou colunas ausentes: {faltantes}"
        )

    ids = pd.to_numeric(tabela["PLAYER_ID"], errors="coerce")

    if not (ids.notna() & ids.gt(0) & ids.mod(1).eq(0)).all():
        raise ValueError(f"{temporada}: identificadores inválidos.")

    if not tabela["SEASON"].eq(temporada).all():
        raise ValueError(f"{temporada}: temporada incorreta ou ausente.")

    if tabela.duplicated(["PLAYER_ID", "SEASON"]).any():
        raise ValueError(f"{temporada}: pares jogador/temporada duplicados.")

    totais = tabela[["FGA", "FGM", "FG3A", "FG3M"]].apply(
        pd.to_numeric, errors="coerce"
    )

    if (
        totais.isna().any().any()
        or totais.lt(0).any().any()
        or totais.mod(1).ne(0).any().any()
    ):
        raise ValueError(f"{temporada}: totais inválidos.")

    if (
        totais["FGM"].gt(totais["FGA"]).any()
        or totais["FG3A"].gt(totais["FGA"]).any()
        or totais["FG3M"].gt(totais["FG3A"]).any()
        or totais["FG3M"].gt(totais["FGM"]).any()
        or (totais["FGM"] - totais["FG3M"]).gt(
            totais["FGA"] - totais["FG3A"]
        ).any()
    ):
        raise ValueError(f"{temporada}: totais inconsistentes.")

    if arquivo_novo:
        temporario = arquivo.with_suffix(".tmp")
        tabela.to_csv(temporario, index=False)
        temporario.replace(arquivo)

    tabelas_jogadores.append(tabela)
    print(f"{temporada}: {len(tabela)} jogadores.")

base_esperada = pd.concat(tabelas_jogadores, ignore_index=True)

print("Total de pares jogador/temporada:", len(base_esperada))

2015-16: 476 jogadores.
2016-17: 486 jogadores.
2017-18: 540 jogadores.
2018-19: 530 jogadores.
2019-20: 529 jogadores.
2020-21: 540 jogadores.
2021-22: 605 jogadores.
2022-23: 539 jogadores.
2023-24: 572 jogadores.
2024-25: 569 jogadores.
Total de pares jogador/temporada: 5386


In [3]:
# MÓDULO 3 — PLANEJAMENTO DA COLETA POR JOGADOR E TEMPORADA
# Organiza os pares únicos que serão consultados.

pares_coleta = (
    base_esperada[["PLAYER_ID", "SEASON"]]
    .drop_duplicates()
    .sort_values(["SEASON", "PLAYER_ID"])
    .reset_index(drop=True)
)

print("Pares planejados:", len(pares_coleta))
display(pares_coleta.groupby("SEASON").size().rename("JOGADORES"))

Pares planejados: 5386


SEASON
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
2020-21    540
2021-22    605
2022-23    539
2023-24    572
2024-25    569
Name: JOGADORES, dtype: int64

In [1]:
# MÓDULO 4 — COLETA INCREMENTAL DOS ARREMESSOS
# Preserva arquivos existentes, registra tentativas e permite retomada.
# A validade dos arquivos será conferida no módulo 7.

inicio = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
arquivo_log = pasta_arremessos / f"log_{inicio}.csv"

registros = []
falhas_consecutivas = 0

for par in pares_coleta.itertuples(index=False):
    player_id = int(par.PLAYER_ID)
    temporada = par.SEASON

    pasta = pasta_arremessos / temporada
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / f"{player_id}.csv"

    if arquivo.exists():
        continue

    sucesso = False

    for tentativa in range(1, 4):
        time.sleep(1)

        try:
            resposta = shotchartdetail.ShotChartDetail(
                player_id=player_id,
                team_id=0,
                season_nullable=temporada,
                season_type_all_star="Regular Season",
                context_measure_simple="FGA",
                timeout=60
            )

            tabela = resposta.shot_chart_detail.get_data_frame()
            tabela["SEASON"] = temporada

        except (RequestException, ValueError) as erro:
            registros.append({
                "PLAYER_ID": player_id,
                "SEASON": temporada,
                "TENTATIVA": tentativa,
                "STATUS": "falha",
                "LINHAS": None,
                "ERRO": str(erro),
                "DATA_UTC": datetime.now(timezone.utc).isoformat()
            })

            pd.DataFrame(registros).to_csv(arquivo_log, index=False)

            if tentativa < 3:
                time.sleep(15 * tentativa)

            continue

        temporario = arquivo.with_suffix(".tmp")
        tabela.to_csv(temporario, index=False)
        temporario.replace(arquivo)

        registros.append({
            "PLAYER_ID": player_id,
            "SEASON": temporada,
            "TENTATIVA": tentativa,
            "STATUS": "salvo" if len(tabela) else "resposta_vazia",
            "LINHAS": len(tabela),
            "ERRO": "",
            "DATA_UTC": datetime.now(timezone.utc).isoformat()
        })

        pd.DataFrame(registros).to_csv(arquivo_log, index=False)

        print(f"{temporada} | {player_id}: {len(tabela)} arremessos.")
        sucesso = True
        break

    falhas_consecutivas = 0 if sucesso else falhas_consecutivas + 1

    if falhas_consecutivas >= 3:
        print("Três pares consecutivos falharam. Pausa de 15 minutos.")
        time.sleep(15 * 60)
        falhas_consecutivas = 0

print("Coleta encerrada.")

if registros:
    print("Log:", arquivo_log.resolve())
else:
    print("Todos os pares já possuíam arquivos.")

NameError: name 'datetime' is not defined